In [1]:
!pip install pandas

In [ ]:
import pandas as pd

plans = pd.read_csv("D:\\31 days challenge\\data\\plans.csv")
claims = pd.read_csv("D:\\31 days challenge\\data\\claims.csv")

# Inspect
print(claims.info())
print(claims.isnull().sum())
print(claims.head())
print(plans.info())
print(plans.head())



# Fix header whitespace
plans.columns = plans.columns.str.strip()
claims.columns = claims.columns.str.strip()

print("plans:", plans.columns.tolist())
print("claims:", claims.columns.tolist())

# Clean
claims = claims.drop_duplicates()
claims["date_filed"] = pd.to_datetime(claims["date_filed"], errors="coerce")
claims["status"] = claims["status"].str.strip().str.title()




<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   claim_id      5 non-null      str  
 1   member_id     5 non-null      str  
 2   plan_id       5 non-null      str  
 3   procedure     5 non-null      str  
 4   claim_amount  5 non-null      int64
 5   status        5 non-null      str  
 6   date_filed    5 non-null      str  
dtypes: int64(1), str(6)
memory usage: 412.0 bytes
None
claim_id        0
member_id       0
plan_id         0
procedure       0
claim_amount    0
status          0
date_filed      0
dtype: int64
  claim_id member_id plan_id procedure  claim_amount    status  date_filed
0    C1001     M1001    P101     X-ray           250   Pending  2023-04-01
1    C1002     M1001    P101   Surgery          1200  Approved  2023-03-15
2    C1003     M1002    P102     X-ray           150    Denied  2023-04-05
3    C1004     M1002    P102   Surgery         

In [5]:
# Join + filter practice
merged = claims.merge(plans, on="plan_id", how="left")
print(merged[merged["claim_amount"] > 200][["claim_id", "plan_name", "claim_amount"]])

  claim_id   plan_name  claim_amount
0    C1001    Gold PPO           250
1    C1002    Gold PPO          1200
3    C1004  Silver HMO           900


###### Create coverage.db with Python's sqlite3 module and load the cleaned DataFrames via df.to_sql() (tables for plans and claims).

In [6]:
import sqlite3

conn = sqlite3.connect("coverage.db")          # creates coverage.db in your repo root
plans.to_sql("plans", conn, if_exists="replace", index=False)
claims.to_sql("claims", conn, if_exists="replace", index=False)

# Verify both tables loaded
print(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn))
print(pd.read_sql("SELECT COUNT(*) FROM plans", conn))   # expect 3
print(pd.read_sql("SELECT COUNT(*) FROM claims", conn))  # expect 5

     name
0   plans
1  claims
   COUNT(*)
0         3
   COUNT(*)
0         5


###### Write and test 5 SQL queries mapped to realistic member questions: - "What's the deductible on the Gold PPO plan?" - "How many claims are pending for member M1001?" - "Which plans have a monthly premium under $400?" - A JOIN between claims and plans - A top-N query (e.g. most claimed procedures)

In [7]:
# Q1: "What's the deductible on the Gold PPO plan?"
q1 = "SELECT plan_name, annual_deductible FROM plans WHERE plan_name = 'Gold PPO';"

# Q2: "How many claims are pending for member M1001?"
q2 = """SELECT COUNT(*) AS pending_claims
FROM claims
WHERE member_id = 'M1001' AND status = 'Pending';"""

# Q3: "Which plans have a monthly premium under $400?"
q3 = "SELECT plan_name, monthly_premium FROM plans WHERE monthly_premium < 400;"

# Q4: JOIN — "Show each claim with its plan details"
q4 = """SELECT c.claim_id, c.procedure, c.claim_amount, c.status,
       p.plan_name, p.coverage_type
FROM claims c
JOIN plans p ON c.plan_id = p.plan_id;"""

# Q5: Top-N — "Most claimed procedures by total amount"
q5 = """SELECT procedure, COUNT(*) AS num_claims, SUM(claim_amount) AS total_amount
FROM claims
GROUP BY procedure
ORDER BY total_amount DESC
LIMIT 3;"""

for i, q in enumerate([q1, q2, q3, q4, q5], 1):
    print(f"--- Q{i} ---")
    print(pd.read_sql(q, conn))

--- Q1 ---
  plan_name  annual_deductible
0  Gold PPO               2000
--- Q2 ---
   pending_claims
0               1
--- Q3 ---
    plan_name  monthly_premium
0  Silver HMO              300
1  Bronze HMO              150
--- Q4 ---
  claim_id procedure  claim_amount    status   plan_name coverage_type
0    C1001     X-ray           250   Pending    Gold PPO           PPO
1    C1002   Surgery          1200  Approved    Gold PPO           PPO
2    C1003     X-ray           150    Denied  Silver HMO           HMO
3    C1004   Surgery           900  Approved  Silver HMO           HMO
4    C1005     X-ray            50   Pending  Bronze HMO           HMO
--- Q5 ---
  procedure  num_claims  total_amount
0   Surgery           2          2100
1     X-ray           3           450


##### Saving the output in the markdown format

In [9]:
!pip install tabulate

In [10]:
queries = {
    "Q1: What's the deductible on the Gold PPO plan?": q1,
    "Q2: How many claims are pending for member M1001?": q2,
    "Q3: Which plans have a monthly premium under $400?": q3,
    "Q4: Show each claim with its plan details (JOIN)": q4,
    "Q5: Most claimed procedures by total amount (Top-N)": q5,
}

with open("structured_queries.md", "w") as f:
    f.write("# Structured Queries — Day 4\n\n")
    for title, q in queries.items():
        df = pd.read_sql(q, conn)
        f.write(f"## {title}\n\n```sql\n{q.strip()}\n```\n\n**Output:**\n\n")
        f.write(df.to_markdown(index=False))
        f.write("\n\n")

print("structured_queries.md written")

structured_queries.md written
